<a href="https://colab.research.google.com/github/SathwikaRevoori/Political-Intelligence-Agent/blob/main/RAGagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
print("Yes")%pip

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Yes


In [ ]:
%pip install --quiet langchain-google-genai langchain-community langchain

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
llm=ChatGoogleGenerativeAI(model="models/gemma-4-31b-it",google_api_key=GOOGLE_API_KEY)

In [ ]:
from langchain_core.documents import Document

big_paragraph = """World War III is often imagined as a large-scale global conflict involving the world’s most powerful nations. Unlike previous world wars, a third world war would likely be far more destructive due to advanced technology, nuclear weapons, cyber warfare, and artificial intelligence. The consequences would not only affect soldiers on battlefields but also civilians, economies, and the environment on an unprecedented scale.

One of the major fears surrounding World War III is the use of nuclear weapons. Even a limited nuclear exchange could cause massive loss of life, long-term radiation damage, and a “nuclear winter” that disrupts global agriculture. In addition, modern warfare would extend beyond physical attacks. Cyber warfare could target critical infrastructure such as power grids, banking systems, and communication networks, causing chaos without a single missile being fired.

Another important aspect is the role of alliances and geopolitics. Conflicts between major powers could quickly escalate if allied nations become involved. Tensions over resources, territory, or ideological differences might act as triggers. Unlike in the past, globalization means that economies are deeply interconnected, so a war would lead to worldwide economic collapse, affecting even countries not directly involved.

However, the possibility of such destruction also acts as a deterrent. Many nations are aware that a full-scale war would be mutually devastating, which encourages diplomacy, negotiations, and international cooperation. Organizations like the United Nations play a role in preventing conflicts and promoting peace.

In conclusion, World War III remains a frightening possibility, but it is not inevitable. The future depends on global leadership, responsible decision-making, and efforts to resolve conflicts peacefully. Avoiding such a catastrophe is one of humanity’s greatest challenges and responsibilities.
"""

documents = Document(page_content=big_paragraph)

print(documents)

page_content='World War III is often imagined as a large-scale global conflict involving the world’s most powerful nations. Unlike previous world wars, a third world war would likely be far more destructive due to advanced technology, nuclear weapons, cyber warfare, and artificial intelligence. The consequences would not only affect soldiers on battlefields but also civilians, economies, and the environment on an unprecedented scale.

One of the major fears surrounding World War III is the use of nuclear weapons. Even a limited nuclear exchange could cause massive loss of life, long-term radiation damage, and a “nuclear winter” that disrupts global agriculture. In addition, modern warfare would extend beyond physical attacks. Cyber warfare could target critical infrastructure such as power grids, banking systems, and communication networks, causing chaos without a single missile being fired.

Another important aspect is the role of alliances and geopolitics. Conflicts between major pow

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=text_splitter.split_documents([documents])
for i,chunk in enumerate(chunks):
  print(f"chunk{i}")
  print(chunk.page_content)

chunk0
World War III is often imagined as a large-scale global conflict involving the world’s most powerful nations. Unlike previous world wars, a third world war would likely be far more destructive due to advanced technology, nuclear weapons, cyber warfare, and artificial intelligence. The consequences would not only affect soldiers on battlefields but also civilians, economies, and the environment on an unprecedented scale.
chunk1
One of the major fears surrounding World War III is the use of nuclear weapons. Even a limited nuclear exchange could cause massive loss of life, long-term radiation damage, and a “nuclear winter” that disrupts global agriculture. In addition, modern warfare would extend beyond physical attacks. Cyber warfare could target critical infrastructure such as power grids, banking systems, and communication networks, causing chaos without a single missile being fired.
chunk2
Another important aspect is the role of alliances and geopolitics. Conflicts between majo

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain.tools import tool
from langchain.agents import create_agent
embeddings=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",google_api_key=GOOGLE_API_KEY)

embedding_dim=len(embeddings.embed_query("Hello world"))
index=faiss.IndexFlatL2(embedding_dim)
docstore=InMemoryDocstore()
vector_store=FAISS(embeddings,index,docstore,index_to_docstore_id={})
vector_store.add_documents(chunks)


['a978386b-4abc-484e-8731-21ff37193594',
 '2c9a8adc-1a95-4dfa-aef2-24b52e7b82e2',
 'f4bb0394-8840-499b-8122-60a708c1aac8',
 'fdc64520-8e27-41b0-a904-3763115e5b9c',
 '0d813b2e-7268-4d93-b9ed-99e297049efa']

In [ ]:
from langchain.tools import tool
@tool(response_format="content_and_artifact")
def retrieve_internet_context(query:str):
  """Retrieve informationn from the internet knowledge base to help answer a query"""
  retrieved_docs=vector_store.similarity_search(query,k=2)
  serialized="\n\n".join(
      (f"source: {doc.metadata}\nContent: {doc.page_content}")
      for doc in retrieved_docs
  )
  return serialized,retrieved_docs

In [ ]:
from langchain.agents import create_agent
tools =[retrieve_internet_context]
prompt=(
    "You are agent that knows world politics and current affairs"
    "Say I dont know, if there is no relevant information"
)
internet_agent = create_agent(llm,tools,system_prompt=prompt)
query = "What is the biggest fear for world war III?"
for event in internet_agent.stream(
 {"messages":[{"role":"user","content":query}]},
 stream_mode="values",
):
  message=event["messages"][-1]
  if isinstance(message.content,list):
    filtered_content = [c for c in message.content if c.get("type")!="thinking"]
    if filtered_content:
      message.content = filtered_content
      message.pretty_print()
  else:
    message.pretty_print()

================================ Human Message =================================

What is the biggest fear for world war III?
================================== Ai Message ==================================

[{'type': 'text', 'text': 'When political scientists, military strategists, and historians discuss the "biggest fear" regarding a third World War, the concerns are usually categorized into three levels: **existential, systemic, and humanitarian.**\n\nWhile there are many fears, the overarching "biggest" fear is **Nuclear Annihilation**, but the modern context has added several other critical layers.\n\n### 1. The Existential Fear: Nuclear Winter\nThe primary fear is not just the initial explosions of nuclear weapons, but the aftermath. \n*   **Mutually Assured Destruction (MAD):** The doctrine that any nuclear attack would result in the total annihilation of both the attacker and the defender.\n*   **Nuclear Winter:** Scientific models suggest that the soot and smoke from burning c

In [ ]:
from google.colab import userdata
NEWS_API_KEY=userdata.get('your_api_key')
print(repr(NEWS_API_KEY))

'949f5be2b39b4402bb96dde8b91305f9'


In [ ]:
pip install requests langchain faiss-cpu langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00


In [ ]:
import requests



def fetch_news(topic):
    url = (
        f"https://newsapi.org/v2/everything?"
        f"q={topic}"
        f"&language=en"
        f"&sortBy=publishedAt"
        f"&pageSize=10"
        f"&apiKey={NEWS_API_KEY}"
    )

    response = requests.get(url)
    data = response.json()

    if data["status"] != "ok":
        raise Exception(data)

    return data["articles"]

result = fetch_news("China Taiwan tensions")
print(result)

[{'source': {'id': None, 'name': 'Crypto Briefing'}, 'author': 'Editorial Team', 'title': 'Taiwanese investors borrow heavily to chase TSMC-driven AI stock rally', 'description': "Heavy borrowing by Taiwanese investors to capitalize on AI stocks could lead to significant market volatility if TSMC's growth falters.\nThe post Taiwanese investors borrow heavily to chase TSMC-driven AI stock rally appeared first on Crypto Briefing.", 'url': 'https://cryptobriefing.com/taiwan-margin-debt-tsmc-ai-rally/', 'urlToImage': 'https://static.cryptobriefing.com/wp-content/uploads/2026/06/22205105/tsmc-1-800x420.jpeg', 'publishedAt': '2026-06-23T00:51:07Z', 'content': "Margin debt has surged over $13 billion to levels not seen since September 2000, raising bubble concerns as Taiwan's market hits 25-year highs Taiwan’s retail investors are loading up on borrowed mon… [+2484 chars]"}, {'source': {'id': None, 'name': 'Ibtimes.com.au'}, 'author': 'Tony Jackson', 'title': 'Nvidia Stock Steady Near $210 as

In [ ]:
all_articles = []

for topic in topics:
    articles = fetch_news(topic)
    all_articles.extend(articles)
print(len(all_articles))
print(type(all_articles[0]))

48
<class 'dict'>


In [ ]:
topics = [
    "Russia Ukraine war",
    "China Taiwan tensions",
    "Israel Gaza conflict",
    "US China relations",
    "NATO expansion"
]

all_articles = []

for topic in topics:
    all_articles.extend(fetch_news(topic))

print(len(all_articles))

Calling API...
Status Code: 200
Response Text:
{"status":"ok","totalResults":3027,"articles":[{"source":{"id":"associated-press","name":"Associated Press"},"author":"Jill Lawless","title":"Ever since Brexit, British politics has been an unruly mess","description":"Brexit fractured the European Union, and it broke British politics. The country is about to get its seventh prime minister since the 2016 vote to leave the EU. Labour Prime Minister Keir Starmer announced Monday he will step down after two years of economic s…","url":"https://apnews.com/article/brexit-vote-10th-anniversary-broken-uk-politics-e9c14876e5e9a51fb57ba6d67e30582f","urlToImage":"https://dims.apnews.com/dims4/default/7b1f55f/2147483647/strip/true/crop/3323x2214+0+0/resize/980x653!/quality/90/?url=https%3A%2F%2Fassets.apnews.com%2F2b%2F0b%2F93c83244012aff0faa55b48cf82e%2F33a26eb4853f49d1a3184ad35b914904","publishedAt":"2026-06-23T06:24:46Z","content":"LONDON (AP) Brexit fractured the European Union, and broke British 

In [ ]:
print(type(all_articles))
print(len(all_articles))

for i in range(min(5, len(all_articles))):
    print(i, type(all_articles[i]), all_articles[i])

<class 'list'>
15
0 <class 'str'> status
1 <class 'str'> totalResults
2 <class 'str'> articles
3 <class 'str'> status
4 <class 'str'> totalResults


In [ ]:
from langchain_core.documents import Document

documents = []

for article in all_articles:
    documents.append(
        Document(
            page_content=f"""
Title: {article.get('title', '')}

Description:
{article.get('description', '')}

Content:
{article.get('content', '')}
""",
            metadata={
                "source": article.get("source", {}).get("name", "Unknown"),
                "url": article.get("url", "")
            }
        )
    )

print(len(documents))

48


In [ ]:
!pip install -q langchain-text-splitters langchain_community

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
73


In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

/tmp/ipykernel_1046/2608617100.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
from langchain.tools import tool

@tool
def retrieve_news_context(query: str):
    """
    Retrieve relevant geopolitical news.
    """

    docs = vector_store.similarity_search(
        query,
        k=5
    )

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

llm = ChatGoogleGenerativeAI(
    model="models/gemma-4-31b-it",
    google_api_key=GEMINI_API_KEY
)

system_prompt = """
You are a geopolitical intelligence analyst.

Use the retrieval tool whenever needed.

Provide:
1. Summary
2. Key actors
3. Risks
4. Possible future developments

If information is insufficient,
say 'I don't know'.
"""

agent = create_agent(
    llm,
    [retrieve_news_context],
    system_prompt=system_prompt
)

In [ ]:
query = "What are the biggest risks surrounding Taiwan?"

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ]
    }
)

print(response)

{'messages': [HumanMessage(content='What are the biggest risks surrounding Taiwan?', additional_kwargs={}, response_metadata={}, id='87ef80e1-4f22-4a80-a874-76e976c84903'), AIMessage(content=[{'type': 'thinking', 'thinking': 'Biggest risks surrounding Taiwan.\nGeopolitical intelligence analyst.\nSummary, Key actors, Risks, Possible future developments.\nUse retrieval tools if needed (though general knowledge is extensive, current events are crucial).\n\n    *   *Core Conflict:* China (PRC) vs. Taiwan (ROC) vs. USA.\n    *   *China\'s Goal:* Reunification (by force if necessary).\n    *   *Taiwan\'s Goal:* Maintaining status quo/autonomy.\n    *   *US Goal:* Strategic ambiguity/stability/democracy/semiconductors.\n    *   *Key Economic Risk:* TSMC/Semiconductors (The "Silicon Shield").\n    *   *Key Geopolitical Risk:* First Island Chain, South China Sea.\n    *   *Immediate Triggers:* Elections, provocative rhetoric, military drills.\n\n    *   *Query:* "current tensions Taiwan China U